### Complete Prediction Pipeline (Using Saved Model)

This section demonstrates how to use the saved model to make predictions on new data

In [8]:
import joblib
import warnings                                       
warnings.filterwarnings('ignore')

# Load preprocessing artifacts
preprocessing_data = joblib.load(
    "../models/obesity_preprocessing.pkl"
)

print("Preprocessing artifacts loaded successfully")

print("\nLoaded artifacts:")
for artifact in preprocessing_data:
    print(f"  • {artifact}")

print("\nProduction environment initialized successfully")

Preprocessing artifacts loaded successfully

Loaded artifacts:
  • scaler
  • target_encoder
  • caec_mapping
  • calc_mapping
  • feature_names
  • target_mapping

Production environment initialized successfully


In [9]:
model = joblib.load("../models/obesity_rf_model.pkl")

scaler = preprocessing_data["scaler"]
target_encoder = preprocessing_data["target_encoder"]

caec_mapping = preprocessing_data["caec_mapping"]
calc_mapping = preprocessing_data["calc_mapping"]

feature_names = preprocessing_data["feature_names"]

In [10]:
import pandas as pd

def preprocess_new_data(
    new_data,
    scaler,
    feature_names,
    caec_mapping,
    calc_mapping
):
    # Gender Encoding
    gender_mapping = {
        "Female": 0,
        "Male": 1
    }

    new_data["Gender"] = new_data["Gender"].map(
        gender_mapping
    )

    # CAEC Encoding
    new_data["CAEC"] = new_data["CAEC"].map(
        caec_mapping
    )

    # CALC Encoding
    new_data["CALC"] = new_data["CALC"].map(
        calc_mapping
    )

    # One-Hot Encoding for MTRANS
    new_data = pd.get_dummies(
        new_data,
        columns=["MTRANS"]
    )

    # Match training columns
    new_data = new_data.reindex(
        columns=feature_names,
        fill_value=0
    )

    # Scaling
    new_data_scaled = scaler.transform(
        new_data
    )

    return new_data_scaled

In [16]:
new_person = {
    'Gender': 'Male',
    'Age': 25.0,
    'Height': 1.75,
    'Weight': 85.0,
    'family_history_with_overweight': 0,
    'FAVC': 1,
    'FCVC': 1.5,
    'NCP': 3.0,
    'CAEC': 'Sometimes',
    'SMOKE': 0,
    'CH2O': 2.0,
    'SCC': 0,
    'FAF': 3.0,
    'TUE': 2.0,
    'CALC': '0',
    'MTRANS': 'Public_Transportation'
}
new_data = pd.DataFrame([new_person])

In [12]:
new_data

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS
0,Male,25.0,1.75,85.0,0,1,1.5,3.0,Sometimes,0,2.0,0,3.0,2.0,0,Public_Transportation


In [13]:
new_data_scaled = preprocess_new_data(
    new_data,
    scaler,
    feature_names,
    caec_mapping,
    calc_mapping
)

In [14]:
prediction = model.predict(new_data_scaled)

predicted_class = target_encoder.inverse_transform(prediction)

print(f"Predicted Obesity Level: {predicted_class[0]}")

Predicted Obesity Level: Overweight_Level_II


In [15]:
probabilities = model.predict_proba(new_data_scaled)

for class_name, probability in zip(target_encoder.classes_,probabilities[0]):
    print(f"{class_name}: {probability:.2%}")

0rmal_Weight: 26.96%
Insufficient_Weight: 0.60%
Obesity_Type_I: 20.15%
Obesity_Type_II: 2.34%
Obesity_Type_III: 0.00%
Overweight_Level_I: 20.58%
Overweight_Level_II: 29.36%


##  Summary & Conclusions

### What We Accomplished:

✓ **Understood the Problem**: Predicting obesity risk levels helps with early intervention

✓ **Analyzed the Dataset**: 20,758 records with 17 health/lifestyle features

✓ **Explained Each Feature**: Why each column matters for obesity prediction

✓ **Preprocessed Data**: Encoded categorical variables and scaled numerical features

✓ **Built Random Forest Model**: Simple yet powerful ensemble classifier

✓ **Evaluated Performance**: Achieved high accuracy with detailed metrics

✓ **Identified Important Features**: Shows which factors matter most

✓ **Saved the Model**: Pickle format for easy reuse and deployment

✓ **Created Prediction Pipeline**: Ready for real-world predictions

### Key Takeaways:

1. **Machine Learning Process**: Data → Preprocessing → Model → Evaluation → Deployment

2. **Random Forest Advantages**:
   - Handles mixed data types naturally
   - Provides feature importance rankings
   - Robust to overfitting
   - No feature scaling required (but we did it anyway)

3. **Model Persistence**: Always save trained models to avoid retraining

4. **Evaluation Metrics**: Accuracy alone isn't enough; use precision, recall, F1-score

### Next Steps for Improvement:

- Hyperparameter tuning (GridSearchCV or RandomizedSearchCV)
- Try other algorithms (Gradient Boosting, Neural Networks)
- Gather more data for better generalization
- Deploy as a web service (Flask, FastAPI)
- Monitor model performance in production
- Collect feedback and retrain periodically

---

# Author
### <a href="https://www.linkedin.com/in/salma-belaicha-04a646336/" target="_blank">Salma Belaicha</a>